In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 972a4a6a-39c3-401d-aa38-582822a29732, 3, Finished, Available, Finished, False)

In [2]:
table_name = "dbo.sales"


StatementMeta(, 972a4a6a-39c3-401d-aa38-582822a29732, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

schema = StructType([
    StructField("SalesOrderNumber", StringType(), True),
    StructField("SalesOrderLineNumber", IntegerType(), True),
    StructField("OrderDate", DateType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("EmailAddress", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("TaxAmount", DoubleType(), True)
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .schema(schema)
    .csv("Files/new_data/*.csv")
)

df_transformed = (
    df_raw
    .withColumn("Year", F.year("OrderDate").cast("int"))
    .withColumn("Month", F.month("OrderDate").cast("int"))
    .withColumn("FirstName", F.substring_index("CustomerName", " ", 1))
    .withColumn("LastName", F.trim(F.expr("substring(CustomerName, length(substring_index(CustomerName, ' ', 1)) + 2)")))
    .select(
        "SalesOrderNumber",
        "SalesOrderLineNumber",
        "OrderDate",
        "Year",
        "Month",
        "FirstName",
        "LastName",
        "EmailAddress",
        "Item",
        "Quantity",
        "UnitPrice",
        "TaxAmount"
    )
    .dropDuplicates()
)

(
    df_transformed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

row_count = spark.table(table_name).count()
notebookutils.notebook.exit(str(row_count))


StatementMeta(, 972a4a6a-39c3-401d-aa38-582822a29732, 5, Finished, Available, Finished, False)

ExitValue: 32718